In [1]:
import pandas as pd
from scipy.sparse import csr_matrix
import implicit
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

In [2]:
meta_df = pd.read_parquet("../data/output/meta_data.parquet")
user_item_itera = pd.read_parquet("../data/output/user-item-interaction.parquet")

In [3]:
user_item_itera.columns

Index(['review_id', 'asin', 'parent_asin', 'user_id', 'review_datetime',
       'review_title', 'review_text', 'review_images', 'verified_purchase',
       'helpful_vote', 'review_rating', 'review_text_length',
       'review_title_length', 'review_word_count', 'is_extreme_rating',
       'is_positive', 'is_negative', 'num_review_img', 'item_title',
       'main_category', 'categories', 'description', 'features', 'details',
       'item_images', 'item_videos', 'item_rating', 'store', 'price',
       'num_item_img', 'num_item_videos', 'is_free', 'has_price_info',
       'price_bucket', 'num_categories', 'days_since_review', 'review_year',
       'review_month', 'recency_weight'],
      dtype='str')

In [4]:
# Select relevant columns
norm_user_item_itera = user_item_itera[[
    "user_id",
    "parent_asin", "num_item_img", "num_item_videos", "price",
    "helpful_vote", "recency_weight", "review_word_count", "num_review_img", "review_rating"
]].copy()

# Normalize feature columns (Min-Max scaling)
cols_to_norm = [
    col for col in norm_user_item_itera.columns
    if col not in ["user_id", "parent_asin"]
]

scaled = scaler.fit_transform(norm_user_item_itera[cols_to_norm])
norm_user_item_itera[cols_to_norm] = scaled

# Invert price explicitly inside feature list
norm_user_item_itera["price"] = 1 - norm_user_item_itera["price"]

# Compute interaction score (unweighted mean, each feature contributes equally)
norm_user_item_itera["interaction"] = norm_user_item_itera[cols_to_norm].mean(axis=1)

# Aggregate to single score per (user, item)
norm_user_item_itera = (
    norm_user_item_itera
    .groupby(["user_id","parent_asin"], as_index=False)["interaction"].mean()
)

norm_user_item_itera

,user_id,parent_asin,interaction
0,ae2222frpdmnomyomcwiantxp7uq,b002zf31nq,0.275388
1,ae22232ob6s7uac75jufyrgbshiq,b00ig2dokm,0.269761
2,ae22236afrrsmqikgg7tptb75qea,b006cq8tc2,0.265416
3,ae22236afrrsmqikgg7tptb75qea,b00b2v66vs,0.141672
4,ae22236afrrsmqikgg7tptb75qea,b00b7s5fdg,0.255419
...,...,...,...
4828475,ahzzzy4dflawpbqyfqfwvacngura,b009hkl4b8,0.265977
4828476,ahzzzy4dflawpbqyfqfwvacngura,b00ab7hesi,0.222515
4828477,ahzzzy4dflawpbqyfqfwvacngura,b014rgfc0k,0.235818
4828478,ahzzzy4dflawpbqyfqfwvacngura,b017s11xmc,0.268066


In [5]:
# Create categorical codes
user_cats = norm_user_item_itera["user_id"].astype("category")
item_cats = norm_user_item_itera["parent_asin"].astype("category")

norm_user_item_itera["user_idx"] = user_cats.cat.codes
norm_user_item_itera["item_idx"] = item_cats.cat.codes

# Mapping dictionaries
idx_to_user = dict(enumerate(user_cats.cat.categories))
idx_to_item = dict(enumerate(item_cats.cat.categories))

# Sparse matrix
num_users = len(user_cats.cat.categories)
num_items = len(item_cats.cat.categories)

user_item_matrix = csr_matrix(
    (
        norm_user_item_itera["interaction"],
        (norm_user_item_itera["user_idx"], norm_user_item_itera["item_idx"])
    ),
    shape=(num_users, num_items)
).astype("float32")

user_item_matrix

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 4828480 stored elements and shape (2589466, 89246)>

In [6]:
# Train ALS
model = implicit.als.AlternatingLeastSquares(factors=50, regularization=0.01, iterations=20)
model.fit(user_item_matrix)  # transpose: ALS expects item-user

c:\Users\user\OneDrive\Desktop\Recommender\.venv\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [13]:
# Get the *categorical code* for the user
some_user_idx = norm_user_item_itera["user_idx"].iloc[0]

# Recommend
recomm_idx, recomm_scores = model.recommend(
    userid=some_user_idx,
    user_items=user_item_matrix[some_user_idx],
    N=5
)

recomm_asins = [idx_to_item[i] for i in recomm_idx]

print(f"Recommendations for {idx_to_user[some_user_idx]}")

recomm_items = meta_df[meta_df["parent_asin"].isin(recomm_asins)]

recomm_items

Recommendations for ae2222frpdmnomyomcwiantxp7uq


,parent_asin,item_title,main_category,categories,description,features,details,item_images,item_videos,item_rating,store,price
3633,b00uc7dg6q,kindle for mac [download],software,"[education & reference, software, writing & li...",[Kindle for Mac reading app gives users the ab...,[Start reading immediately with three free boo...,"{""Release date"": ""July 29, 2015"", ""Pricing"": ""...","{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",10163.0,amazon digital services inc.,NaN
30605,b007zgo7em,calculator plus free,appstore for android,[],"[*** One of USA TODAY's ""25 essential Kindle F...","[One of USA TODAY's ""25 essential Kindle Fire ...","{""Release Date"": ""2012"", ""Date first listed on...","{'hi_res': [None, None, None, None], 'large': ...","{'title': [], 'url': [], 'user_id': []}",20919.0,"digitalchemy, llc",NaN
32313,b008jgsm6g,flow free,appstore for android,[],[Flow Free® is a simple yet addictive puzzle g...,"[Over 2,500 free puzzles, Free Play and Time T...","{""Release Date"": ""2012"", ""Date first listed on...","{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': [], 'user_id': []}",23129.0,big duck games llc,0.0
44137,b00xzfcvk4,max,appstore for android,[],"[It’s all here. Iconic series, award-winning m...","[Browse or search with ease across HBO, movies...","{""Release Date"": ""2015"", ""Date first listed on...","{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': [], 'user_id': []}",383312.0,"warnermedia global digital services, llc",0.0
50015,b005k17ru0,accuweather with superior accuracy™,appstore for android,[],[Stay connected to the latest weather conditio...,[AccuWeather MinuteCast – minute-by-minute wea...,"{""Release Date"": ""2011"", ""Date first listed on...","{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': [], 'user_id': []}",30249.0,accuweather,0.0


### Save Model

In [ ]:
# import pickle

# with open("cf_model.pkl", "wb") as f:
#     pickle.dump(model, f)

# with open("user_item_matrix.pkl", "wb") as f:
#     pickle.dump(user_item_matrix, f)

###############################################################

# model.save("als_model.npz")

### Load model for inference tool

In [ ]:
# with open("cf_model.pkl", "rb") as f:
#     model = pickle.load(f)

# with open("user_item_matrix.pkl", "rb") as f:
#     user_item_matrix = pickle.load(f)

###############################################################

# model = implicit.als.AlternatingLeastSquares()
# model.load("als_model.npz")